# 📊 Notebook 5 — Full Comparison
> All Four Strategies vs SPY Buy & Hold  |  2020–2022

---

## 1. Overview

| Strategy | Bias | Cost | Vega | Theta |
|---|---|---|---|---|
| Buy-Write | Long, capped | Credit | Short | Long |
| Protective Put | Long, protected | Debit | Long | Short |
| Collar | Neutral-Long | ~Zero | ~Neutral | ~Neutral |
| Long Straddle | Volatility | Debit | Long | Short |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_data
from src.plotting    import *
from src.metrics     import compute_metrics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

DATA_PATH = '../data/spy_simulated_2020_2022.csv'
# Uncomment to use real data:
# DATA_PATH = '../data/spy_2020_2022.csv'

df = load_data(DATA_PATH)
print('Data loaded:', df.shape)

In [ ]:
from src.strategy_buywrite       import build_buywrite
from src.strategy_protective_put  import build_protective_put
from src.strategy_collar          import build_collar
from src.strategy_straddle        import build_straddle
from src.metrics                  import metrics_table
from src.plotting                 import plot_comparison_dashboard

bw  = build_buywrite(df)
pp  = build_protective_put(df)
col = build_collar(df)
std = build_straddle(df)

spy_ref = bw[['spy_bnh','spy_price']].copy()
results_dict = {
    'Buy-Write'      : bw,
    'Protective Put' : pp,
    'Collar'         : col,
    'Long Straddle'  : std,
    'SPY B&H'        : spy_ref,
}
print('All strategies built.')

## 2. Comparison Dashboard

In [ ]:
plot_comparison_dashboard(results_dict,
                          save_path='../outputs/chart_comparison.png')

## 3. Metrics Table

In [ ]:
mt = metrics_table(results_dict)
mt

## 4. Correlation of Daily Returns

In [ ]:
rets = pd.DataFrame({
    n: (res['spy_bnh' if n=='SPY B&H' else 'index_level']).pct_change()
    for n, res in results_dict.items()
})
corr = rets.corr()

fig, ax = plt.subplots(figsize=(8,6), facecolor=BG)
ax.set_facecolor(PANEL)
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=30, color=WHITE, fontsize=9)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index, color=WHITE, fontsize=9)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', color='black', fontsize=8, fontweight='bold')
for s in ax.spines.values(): s.set_edgecolor(GREY)
ax.set_title('Correlation of Daily Returns', color=WHITE, fontsize=11, fontweight='bold')
fig.patch.set_facecolor(BG)
plt.tight_layout(); plt.show()